# Homework 8.1 - Task 8

## LLM Audit of the Reverse-Dynamics Sign

The purpose of this task is not to accept an LLM derivation automatically.

The LLM must state its time convention clearly, and the sign of the score term must be checked independently.

## 1. Prompt given to the LLM

Consider the forward SDE $dX_t=f(X_t,t)dt+g(t)dW_t$ with score $s(x,t)=\partial_x\log p_t(x)$.

Derive both the reverse-time SDE and the probability-flow ODE.

State the formula when the original time variable $t$ is numerically integrated from $T$ down to $0$, and then rewrite it using the increasing reverse clock $\tau=T-t$.

Be explicit about the sign of the score term.

Finally, check the sign on a centered Gaussian whose variance grows forward in time: reverse generation should contract samples toward zero.

## 2. Recorded LLM response

For the forward SDE

$dX_t=f(X_t,t)dt+g(t)dW_t$

with score

$s(x,t)=\partial_x\log p_t(x)$,

the reverse-time SDE written in the original time coordinate and interpreted while integrating $t$ from $T$ down to $0$ has drift

$f(x,t)-g(t)^2s(x,t)$.

Because the numerical time increment is negative in that convention, it is safer to introduce the increasing reverse clock

$\tau=T-t$.

If $Y_\tau=X_{T-\tau}$, then the reverse-clock SDE drift is

$-f(Y_\tau,t)+g(t)^2s(Y_\tau,t)$,

with $t=T-\tau$.

The forward probability-flow ODE is

$dX_t/dt=f(X_t,t)-g(t)^2s(X_t,t)/2$.

Therefore, with the increasing reverse clock,

$dY_\tau/d\tau=-f(Y_\tau,t)+g(t)^2s(Y_\tau,t)/2$.

The sign check is important. For a centered Gaussian with variance increasing forward in time, the score is negative for positive $x$. The correct reverse-clock drift should be negative for positive $x$, so reverse generation contracts the distribution. Reversing the sign of the score term would instead push positive samples farther outward.

## 3. Algebraic audit

The recorded response gives the reverse-clock SDE drift

$$
-f+g^2s
$$

and the reverse-clock probability-flow drift

$$
-f+\frac{1}{2}g^2s.
$$

These agree with the time change

$$
\tau=T-t.
$$

The next cell checks the sign numerically on the centered Gaussian.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw8_1"

config = json.loads(
    (
        DATA_DIR
        / "task1_configuration.json"
    ).read_text(
        encoding="utf-8"
    )
)

Q = float(
    config["q"]
)

In [2]:
def alpha(t):
    return np.cos(
        0.5
        * np.pi
        * t
    )


def variance(t):
    return (
        1.0
        - Q
        * alpha(t)**2
    )


def beta(t):
    return (
        np.pi
        * np.tan(
            0.5
            * np.pi
            * t
        )
    )


def centered_score(
    x,
    t,
):
    return (
        -x
        / variance(t)
    )


def forward_drift(
    x,
    t,
):
    return (
        -0.5
        * beta(t)
        * x
    )


t = 0.5
x = 1.0

f = forward_drift(
    x,
    t,
)

g2 = beta(t)

s = centered_score(
    x,
    t,
)

correct_reverse_clock_pf = (
    -f
    + 0.5
    * g2
    * s
)

wrong_reverse_clock_pf = (
    -f
    - 0.5
    * g2
    * s
)

correct_reverse_clock_sde = (
    -f
    + g2
    * s
)

wrong_reverse_clock_sde = (
    -f
    - g2
    * s
)

audit = pd.DataFrame([
    {
        "formula": (
            "correct reverse-clock probability flow"
        ),
        "drift_at_x_equals_1_t_equals_0_5": (
            correct_reverse_clock_pf
        ),
        "contracts_positive_x": (
            correct_reverse_clock_pf < 0
        ),
    },
    {
        "formula": (
            "wrong score sign probability flow"
        ),
        "drift_at_x_equals_1_t_equals_0_5": (
            wrong_reverse_clock_pf
        ),
        "contracts_positive_x": (
            wrong_reverse_clock_pf < 0
        ),
    },
    {
        "formula": (
            "correct reverse-clock SDE"
        ),
        "drift_at_x_equals_1_t_equals_0_5": (
            correct_reverse_clock_sde
        ),
        "contracts_positive_x": (
            correct_reverse_clock_sde < 0
        ),
    },
    {
        "formula": (
            "wrong score sign SDE"
        ),
        "drift_at_x_equals_1_t_equals_0_5": (
            wrong_reverse_clock_sde
        ),
        "contracts_positive_x": (
            wrong_reverse_clock_sde < 0
        ),
    },
])

audit.to_csv(
    DATA_DIR
    / "task8_llm_reverse_sign_audit.csv",
    index=False,
)

print(
    audit.to_string(
        index=False
    )
)

                               formula  drift_at_x_equals_1_t_equals_0_5  contracts_positive_x
correct reverse-clock probability flow                         -1.385997                  True
     wrong score sign probability flow                          4.527589                 False
             correct reverse-clock SDE                         -4.342790                  True
                  wrong score sign SDE                          7.484382                 False


## 4. Audit conclusion

The recorded LLM response gets the score-term sign correct under both stated time conventions.

The numerical centered-Gaussian check independently confirms the result:

- for positive $x$, the correct reverse-clock probability-flow drift points inward;
- for positive $x$, the correct reverse-clock SDE drift also points inward;
- reversing the score-term sign makes those drifts point outward.

For the reverse SDE, this statement concerns the deterministic drift. Brownian noise remains present, so an individual stochastic sample path is not required to move inward at every instant.

This task also shows why writing only "reverse drift equals ..." without stating the time convention is ambiguous and risky.
